# E02 — Dispatch policies and storage models

| | |
|---|---|
| **Question** | What battery control logic can Vessim express natively, what must be written by hand, and how do its battery models differ? |
| **Hypotheses** | H3 (flexibility management is weak), H4 (energy-side abstractions are cleaner) |
| **Branch** | `exp/e02-dispatch-and-storage` |
| **Site** | Edge study S1.2 inputs (cached in `data/`), 500 kWh battery, day-ahead price of the edge run, 2025-04-01 |

**Variants**
1. Control strategies on the same site and battery (loss-aware `EfficiencyBattery`, edge parameters):

| Strategy | How it is built in Vessim | Needs foresight? |
|---|---|---|
| Greedy (Vessim default) | built-in `DefaultDispatchPolicy` | no |
| Fixed discharge power | built-in `DefaultDispatchPolicy(charge_power=…)` | no |
| Time-of-use windows | built-in policy + custom `Controller` switching `charge_power` | fixed tariff windows |
| Price thresholds | custom `DispatchPolicy` reading the price grid signal | the day's price quantiles |
| Peak shaving | custom `DispatchPolicy` using only the current imbalance | the import threshold |
| Look-ahead, minimum cost / peak | LP solved outside Vessim, replayed via a grid signal | perfect forecast of load and price |

2. Battery models: Vessim `SimpleBattery`, Vessim `ClcBattery` (electrochemical, per cell, scaled to 500 kWh) and the
   custom `EfficiencyBattery` from E01, all following the same requested schedule.
3. Step size: 15, 5 and 1 min for the strategies and for `ClcBattery`.

**Measures:** energy cost, peak grid import, grid import energy, battery throughput and losses, equivalent full
cycles, SOC window respected, lines of custom code, effect of step size.

## 1. Setup

In [1]:
import importlib.metadata as md_
import pandas as pd
import vessim as vs
from IPython.display import display

from vessim_eval import RESULTS_DIR
from vessim_eval import plotting as vp
from vessim_eval.dispatch import (PeakShavingPolicy, PriceThresholdPolicy, TimeOfUseController,
                                  _balance_with_grid, logic_lines, optimal_schedule,
                                  policy_kpis, price_from_reference, run_policy)
from vessim_eval.edge_reference import load_reference, scenario_config
from vessim_eval.scenarios import BatteryParams, SiteInputs
from vessim_eval.vessim_models import SetpointPolicy

print("vessim", md_.version("vessim"))

vessim 0.16.0


In [2]:
# ---- Parameters -------------------------------------------------------------
SCENARIO = "s1_2_battery_pv_large"
STEP_SIZES_S = {"15 min": 900, "5 min": 300, "1 min": 60}
FIXED_DISCHARGE_KW = 50.0                    # constant discharge request (native option)
TOU_WINDOWS = [(4, 7, -250.0),               # (start h, end h, kW; + = charge, − = discharge)
               (10, 13, 250.0),              # cheap midday hours: charge
               (17, 21, -250.0)]             # evening price peak: discharge
PRICE_QUANTILES = (0.25, 0.75)               # charge below / discharge above
PEAK_SWEEP_KW = list(range(5000, 6301, 50))  # threshold sweep for the peak-shaving policy
OUT = RESULTS_DIR / "E02"

site_cfg = scenario_config(SCENARIO)["data_centers"][0]
battery = BatteryParams.from_edge_config(site_cfg["battery_config"])
ref = load_reference("e01_edge_s1_2_15min.csv")
site = SiteInputs.from_reference(ref, site_cfg["hardware"]["base_load"])
price = price_from_reference(ref)                                     # €/MWh
net_load_kw = site.it_kw + site.cooling_kw + site.building_kw - site.pv_kw

price_low, price_high = price.quantile(PRICE_QUANTILES)
lp_cost = optimal_schedule(net_load_kw, price, battery, "cost")       # kW, + = discharge
lp_peak = optimal_schedule(net_load_kw, price, battery, "peak")
peak_threshold_kw = (net_load_kw - lp_peak).max()                     # best achievable peak
pd.Series({"Battery": f"{battery.capacity_kwh:.0f} kWh, ±{battery.max_charge_kw:.0f} kW, "
                      f"SOC {battery.soc_min:.0%}–{battery.soc_max:.0%}, η = {battery.eff_ch:.0%}",
           "Price thresholds (€/MWh)": f"{price_low:.1f} / {price_high:.1f}",
           "Peak-shaving threshold (kW)": f"{peak_threshold_kw:,.1f}",
           "Edge study LP peak (kW)": f"{ref.lp_grid_kw.max():,.1f}"}, name="Value").to_frame()

,Value
Battery,"500 kWh, ±250 kW, SOC 15%–90%, η = 95%"
Price thresholds (€/MWh),67.1 / 118.6
Peak-shaving threshold (kW),"6,005.3"
Edge study LP peak (kW),"6,005.3"


Energy cost is compared **including the change in stored energy** (valued at the day's mean price): strategies end
the day with different battery charge, and the raw cost would reward simply emptying the battery.

The peak-shaving threshold is set to the lowest peak the look-ahead LP can reach. This is the best case for the
peak-shaving policy; section 3.3 sweeps the threshold to show how much that foresight matters. Our LP
reproduces the edge study's own offline-optimal peak (last row).

In [3]:
fig = vp.stacked_panels([
    ("Site net load (kW)", {"Site net load (demand − PV)": net_load_kw}),
    ("Day-ahead price (€/MWh)", {"Day-ahead price": price}),
], title="Inputs: site net load and electricity price, 2025-04-01")
fig.add_hline(y=price_low, line=dict(color="#888", dash="dot"), row=2, col=1,
              annotation_text="charge below", annotation_position="bottom right")
fig.add_hline(y=price_high, line=dict(color="#888", dash="dot"), row=2, col=1,
              annotation_text="discharge above", annotation_position="top right")
fig.show()

In [4]:
def strategies(step_s: int) -> dict:
    """Fresh policy objects per run (policies and controllers are stateful)."""
    tou = vs.DefaultDispatchPolicy(charge_power=TimeOfUseController.IDLE_W)
    return {
        "Greedy (Vessim default)": dict(policy=vs.DefaultDispatchPolicy()),
        "Fixed discharge power": dict(policy=vs.DefaultDispatchPolicy(
            charge_power=-FIXED_DISCHARGE_KW * 1e3)),
        "Time-of-use windows": dict(policy=tou, controllers=[
            lambda p, b: TimeOfUseController(p, TOU_WINDOWS, step_s)]),
        "Price thresholds": dict(policy=PriceThresholdPolicy(price_low, price_high)),
        "Peak shaving": dict(policy=PeakShavingPolicy(peak_threshold_kw)),
        "Look-ahead, minimum cost": dict(policy=SetpointPolicy(), schedule_kw=lp_cost),
        "Look-ahead, minimum peak": dict(policy=SetpointPolicy(), schedule_kw=lp_peak),
    }

## 2. Run

In [5]:
runs, walls = {}, {}
for label, step_s in STEP_SIZES_S.items():
    for name, kw in strategies(step_s).items():
        runs[name, label], walls[name, label] = run_policy(
            site, battery, battery_kind="losses", price=price, step_s=step_s, **kw)

# Battery models: same requested schedule (look-ahead, minimum cost) and the greedy default.
MODELS = {"SimpleBattery (Vessim)": "simple", "EfficiencyBattery (edge parameters)": "losses",
          "ClcBattery (Vessim)": "clc"}
model_runs = {}
for label, step_s in STEP_SIZES_S.items():
    for name, kind in MODELS.items():
        model_runs[name, "schedule", label], walls[name, "schedule", label] = run_policy(
            site, battery, SetpointPolicy(), battery_kind=kind, price=price,
            schedule_kw=lp_cost, step_s=step_s)
        model_runs[name, "greedy", label], walls[name, "greedy", label] = run_policy(
            site, battery, vs.DefaultDispatchPolicy(), battery_kind=kind, price=price,
            step_s=step_s)

# Peak-shaving threshold sweep (15 min).
sweep = {t: run_policy(site, battery, PeakShavingPolicy(t), price=price)[0]
         for t in PEAK_SWEEP_KW}

w = pd.Series(walls)
print(f"{len(w) + len(sweep)} Vessim runs; per run: median {w.median() * 1e3:.0f} ms, "
      f"max {w.max() * 1e3:.0f} ms; total {w.sum():.2f} s")

66 Vessim runs; per run: median 15 ms, max 184 ms; total 1.36 s


## 3. Results

### 3.1 Control strategies (15-min step, loss-aware battery)

In [6]:
NAMES = list(strategies(900))
kpis = pd.DataFrame({n: policy_kpis(runs[n, "15 min"], price, battery) for n in NAMES}).T
base = kpis.loc["Greedy (Vessim default)"]
ADJ = "Cost incl. change in stored energy (€)"
kpis.insert(2, "Adjusted cost vs greedy (€)", kpis[ADJ] - base[ADJ])
kpis.insert(4, "Peak vs greedy (kW)", kpis["Peak grid import (kW)"] - base["Peak grid import (kW)"])
OUT.mkdir(parents=True, exist_ok=True)
kpis.to_csv(OUT / "strategy_kpis.csv")
display(vp.format_table(kpis.drop(columns=["SOC window respected"]), 2))
kpis[["SOC window respected"]].T

,Energy cost (€),Cost incl. change in stored energy (€),Adjusted cost vs greedy (€),Peak grid import (kW),Peak vs greedy (kW),Grid import energy (kWh),Battery charge energy (kWh),Battery discharge energy (kWh),Battery losses (kWh),Equivalent full cycles,Minimum state of charge (%),Maximum state of charge (%),Final state of charge (%)
Greedy (Vessim default),"10,447.17","10,463.14",0.00,"6,255.27",0.00,"109,991.26",0.00,166.25,8.75,0.33,15.00,50.00,15.00
Fixed discharge power,"10,446.93","10,462.90",-0.23,"6,255.27",0.00,"109,991.26",0.00,166.25,8.75,0.33,15.00,50.00,15.00
Time-of-use windows,"10,398.69","10,414.66",-48.47,"6,238.57",-16.70,"110,029.75",394.74,522.50,47.24,1.05,15.00,90.00,15.00
Price thresholds,"10,427.71","10,422.01",-41.13,"6,183.57",-71.70,"110,279.75",644.74,522.50,59.74,1.04,15.00,90.00,62.50
Peak shaving,"10,493.27","10,475.02",11.88,"6,005.27",-250.00,"110,390.64",442.31,209.18,33.13,0.42,45.96,90.00,90.00
"Look-ahead, minimum cost","10,386.09","10,386.09",-77.05,"6,255.27",0.00,"110,234.48",789.47,712.50,76.97,1.42,15.00,90.00,50.00
"Look-ahead, minimum peak","10,451.50","10,451.50",-11.63,"6,005.27",-250.00,"110,180.11",231.78,209.18,22.60,0.42,15.00,59.04,50.00


,Greedy (Vessim default),Fixed discharge power,Time-of-use windows,Price thresholds,Peak shaving,"Look-ahead, minimum cost","Look-ahead, minimum peak"
SOC window respected,True,True,True,True,True,True,True


In [7]:
fig = vp.stacked_panels([
    ("Grid import (kW)", {n: runs[n, "15 min"]["grid_kw"] for n in NAMES}),
    ("Battery power (kW, + = discharge)", {n: runs[n, "15 min"]["battery_kw"] for n in NAMES}),
    ("Battery state of charge (%)", {n: runs[n, "15 min"]["soc"] * 100 for n in NAMES}),
], title="Grid import, battery power and state of charge per control strategy", row_height=300)
vp.save(fig, OUT / "strategies", png=True)
fig.show()

In [8]:
fig = vp.tradeoff_scatter(
    kpis, "Peak grid import (kW)", ADJ,
    title="Energy cost vs peak grid import per control strategy (one day)",
    x_title="Peak grid import (kW)",
    y_title="Energy cost incl. change in stored energy (€)")
vp.save(fig, OUT / "cost_vs_peak", png=True)
fig.show()

### 3.2 Custom code needed

In [9]:
loc = pd.Series({
    "Greedy (Vessim default)": 0,
    "Fixed discharge power": 0,
    "Time-of-use windows": logic_lines(TimeOfUseController),
    "Price thresholds": logic_lines(PriceThresholdPolicy, _balance_with_grid),
    "Peak shaving": logic_lines(PeakShavingPolicy, _balance_with_grid),
    "Look-ahead (LP + replay policy)": logic_lines(optimal_schedule, SetpointPolicy),
}, name="Lines of custom code (no comments or docstrings)")
loc.to_frame()

,Lines of custom code (no comments or docstrings)
Greedy (Vessim default),0
Fixed discharge power,0
Time-of-use windows,11
Price thresholds,28
Peak shaving,26
Look-ahead (LP + replay policy),61


### 3.3 Peak shaving without foresight: threshold sweep

In [10]:
sweep_kpis = pd.DataFrame({t: policy_kpis(df, price, battery) for t, df in sweep.items()}).T
fig = vp.stacked_panels([
    ("Peak grid import reached (kW)", {
        "Peak shaving": sweep_kpis["Peak grid import (kW)"],
        "Threshold": pd.Series(sweep_kpis.index, index=sweep_kpis.index, dtype=float)}),
    ("Cost incl. change in stored energy (€)", {"Peak shaving": sweep_kpis[ADJ]}),
], title="Peak-shaving policy: peak and cost vs chosen import threshold (dotted line: LP optimum)",
   x_title="Import threshold (kW)", shape="linear")
fig.update_traces(mode="lines+markers")
fig.add_vline(x=peak_threshold_kw, line=dict(color="#888", dash="dot"))
vp.save(fig, OUT / "peak_threshold_sweep", png=True)
fig.show()

### 3.4 Battery models under the same requested schedule

In [11]:
def model_table(control: str, label: str = "1 min") -> pd.DataFrame:
    return pd.DataFrame({m: policy_kpis(model_runs[m, control, label], price, battery)
                         for m in MODELS}).T

sched = model_table("schedule")       # 1-min step: ClcBattery is not meant for large steps
dt_h = (lp_cost.index[1] - lp_cost.index[0]).total_seconds() / 3600   # schedule step (15 min)
sched.insert(0, "Requested discharge (kWh)", lp_cost.clip(lower=0).sum() * dt_h)
sched.insert(1, "Requested charge (kWh)", (-lp_cost).clip(lower=0).sum() * dt_h)
display(vp.format_table(sched[["Requested discharge (kWh)", "Battery discharge energy (kWh)",
                               "Requested charge (kWh)", "Battery charge energy (kWh)",
                               "Battery losses (kWh)", "Energy cost (€)",
                               "Minimum state of charge (%)", "Maximum state of charge (%)",
                               "Final state of charge (%)", "SOC window respected"]], 2))

,Requested discharge (kWh),Battery discharge energy (kWh),Requested charge (kWh),Battery charge energy (kWh),Battery losses (kWh),Energy cost (€),Minimum state of charge (%),Maximum state of charge (%),Final state of charge (%),SOC window respected
SimpleBattery (Vessim),712.50,712.50,789.47,789.47,-0.00,"10,386.09",20.86,99.80,65.39,False
EfficiencyBattery (edge parameters),712.50,712.50,789.47,789.47,76.97,"10,386.09",15.00,90.00,50.00,True
ClcBattery (Vessim),712.50,712.49,789.47,769.22,26.90,"10,384.93",16.74,92.18,55.97,False


In [12]:
series = {m: model_runs[m, "schedule", "1 min"]["soc"] * 100 for m in MODELS}
power = {"Requested schedule": lp_cost,
         **{m: model_runs[m, "schedule", "1 min"]["battery_kw"] for m in MODELS}}
fig = vp.stacked_panels([
    ("Battery power (kW, + = discharge)", power),
    ("Battery state of charge (%)", series),
], title="Battery models following the minimum-cost schedule (1-min step; edge SOC window 15–90 %)",
   row_height=300)
vp.save(fig, OUT / "battery_models", png=True)
fig.show()

In [13]:
vp.format_table(model_table("greedy")[["Battery discharge energy (kWh)", "Battery losses (kWh)",
                                       "Energy cost (€)", "Minimum state of charge (%)",
                                       "SOC window respected"]], 2)

,Battery discharge energy (kWh),Battery losses (kWh),Energy cost (€),Minimum state of charge (%),SOC window respected
SimpleBattery (Vessim),175.00,-0.00,"10,446.35",15.00,True
EfficiencyBattery (edge parameters),166.25,8.75,"10,447.17",15.00,True
ClcBattery (Vessim),172.58,2.42,"10,446.58",15.00,True


### 3.5 Step-size sensitivity

In [14]:
rows = {}
for (name, label), df in runs.items():
    k = policy_kpis(df, price, battery)
    rows[name, label] = k[["Energy cost (€)", "Peak grid import (kW)",
                           "Battery discharge energy (kWh)"]]
for (name, control, label), df in model_runs.items():
    if name == "ClcBattery (Vessim)" and control == "schedule":
        k = policy_kpis(df, price, battery)
        rows["ClcBattery, minimum-cost schedule", label] = k[[
            "Energy cost (€)", "Peak grid import (kW)", "Battery discharge energy (kWh)"]]
steps = pd.DataFrame(rows).T
cost = steps["Energy cost (€)"].unstack()[list(STEP_SIZES_S)]
delta = cost.sub(cost["15 min"], axis=0)
display(vp.format_table(steps.unstack()[[(c, l) for c in steps.columns for l in STEP_SIZES_S]], 2))
fig = vp.grouped_bars(delta[["5 min", "1 min"]].round(2) + 0.0,   # + 0.0 drops "-0.0"
                      title="Change in daily energy cost vs a 15-min step",
                      y_title="Cost difference vs 15-min step (€)", horizontal=True,
                      height=520)
vp.save(fig, OUT / "step_size", png=True)
fig.show()

## 4. Evaluation

- **Native control is greedy or constant.** `DefaultDispatchPolicy` empties the battery in the first hour (as in
  E01). Its only option, a fixed `charge_power`, discharges more slowly but gives the same result. Neither lowers
  the peak or the cost. `charge_power = 0` does not mean *idle* — it switches back to greedy dispatch — so idle
  periods need a workaround (a negligible setpoint).
- **Rule-based strategies need little code.** A custom `DispatchPolicy` (price thresholds, peak shaving) is
  ≈ 25–30 lines; a time-of-use `Controller` on top of the built-in policy is 11 lines. Policies can read the price
  signal and the storage SOC, but get no timestamp, no forecast and no individual actor powers.
- **No strategy wins both objectives.** The minimum-cost look-ahead saves ≈ 77 € (0.7 % of the daily bill) but
  keeps the peak; the minimum-peak look-ahead and the peak-shaving rule cut the peak by 250 kW (the battery's power
  limit). Price thresholds and time-of-use windows sit in between.
- **Peak shaving without a forecast reaches the optimum here.** Any threshold at or below the LP optimum gives the
  same 6 005 kW peak, because on this day the battery is power-limited, not energy-limited. The rule recharges to
  90 % at any price, so its cost is ≈ 12 € higher than greedy.
- **Look-ahead must be built outside Vessim.** The LP (≈ 40 lines plus the replay policy) is solved before the run
  and passed in as a grid signal. It reproduces the edge study's offline-optimal peak (6 005.3 kW).
- **Battery models differ in what they enforce.** `SimpleBattery` follows any request without losses and charges
  to 99.8 % (no upper SOC limit). `ClcBattery` adds current-dependent charge acceptance and internal losses
  (≈ 27 kWh vs 77 kWh for the 95 %/95 % edge battery), but also has no upper SOC limit (92 %) and is parameterised
  per cell. Only the custom `EfficiencyBattery` respects the edge SOC window.
- **Step size changes nothing** for the strategies (inputs and schedules are piecewise constant over 15 min, and
  the time-of-use controller decides for the next interval). Only `ClcBattery` depends on it (+0.8 € at 1 min),
  because its charge limit depends on the step duration.
- **Speed:** 66 runs, median ≈ 16 ms per simulated day.